# Exploration

First look at the synthetic wearables dataset: shape, missing values, types, category
counts, encoding and simple correlations with sleep efficiency.

Training and evaluation live in `train.py` and the `sleep_predictor/` package, not here.
The original version of this notebook split rows at random (so the same person appeared in
training and test) and ran Lasso on the full dataset before splitting; both are replaced by
the user-grouped pipeline in `train.py`. Correlations below are pooled across people and
describe associations only.

Download the CSV into `data/` first (see the README).

In [ ]:
import pandas as pd
df = pd.read_csv('../data/wearables_health_6mo_daily.csv')
df.shape

In [ ]:
df

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.dtypes)

In [ ]:
df_cleaned = df.dropna().copy()

In [ ]:
df_cleaned.shape

In [ ]:

df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])

df_cleaned = df_cleaned.sort_values(by=['user_id', 'date']).reset_index(drop=True)


category_cols = ['gender', 'region', 'workout_type', 'mood']
for col in category_cols:
    print(f"\n--- {col} ---")
    print(df_cleaned[col].value_counts())

In [ ]:
cols_to_drop = ['region', 'device_model', 'height_cm', 'weight_kg']
df_encoded = df_cleaned.drop(columns=cols_to_drop).copy()

mood_mapping = {'very_bad': 1, 'bad': 2, 'neutral': 3, 'good': 4, 'very_good': 5}
df_encoded['mood_encoded'] = df_encoded['mood'].map(mood_mapping)
df_encoded = df_encoded.drop(columns=['mood'])

df_encoded = pd.get_dummies(df_encoded, columns=['gender', 'workout_type'], drop_first=True)

print(f"New column count: {df_encoded.shape[1]}")
print(df_encoded.columns.tolist())

In [ ]:
features_to_test = ['steps', 'alcohol_units', 'stress_score', 'caffeine_mg',
                    'workout_type_none', 'screen_time_min', 'workout_type_walk']

correlations = df_encoded[features_to_test + ['sleep_efficiency']].corr()['sleep_efficiency']
print(correlations.sort_values(ascending=False))